# Tutorial 3.2: Simulation of Random Incomplete Registration in Embryonic Mouse Brain

This tutorial examines random incomplete registration in the embryonic mouse brain ATAC-to-RNA setting. ATAC remains complete, whereas RNA is withheld at 50% of locations sampled across the tissue, creating a spatially dispersed target-unregistered pattern rather than a contiguous coverage boundary.

The paired pre-masking RNA profiles provide location-matched ground truth, allowing PRISM to be assessed when observed and RNA-unregistered locations are interspersed throughout the same tissue environments.


In [ ]:
# Environment and imports
from pathlib import Path

import scanpy as sc
import PRISM
from PRISM import (plot_imputation_metric_boxplot, compute_similarity_prior, plot_prism_imputation_spatial,
                   preprocess_omics, prism_eval_and_save, run_clustering_eval_plot, select_best_device,
                   set_prism_plot_style, set_seed, simulate_missing_sliding)
set_prism_plot_style()

In [ ]:
# Load data and set up paths
DEVICE = select_best_device()
RANDOM_SEED = 2024
set_seed(RANDOM_SEED)
Slice_ID = "E15.5" 

DATASET_DIR = Path("Datasets") / "embryonic mouse brain" / Slice_ID
SOURCE_H5AD = DATASET_DIR / "adata_atac.h5ad"
TARGET_H5AD = DATASET_DIR / "adata_rna.h5ad"
RESULTS_DIR = Path("Results") / "Tutorial3_2_embryonic_mouse_brain"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PRIOR_PATH = RESULTS_DIR / f"{Slice_ID}_emb_mouse_AOT.npz"
RUN_PREFIX = f"{Slice_ID}_emb_mouse"

adata_source = sc.read_h5ad(SOURCE_H5AD)
adata_target = sc.read_h5ad(TARGET_H5AD)
adata_source.var_names_make_unique()
adata_target.var_names_make_unique()


### Simulating incomplete registration

This tutorial simulates random incomplete registration by withholding RNA at 50% of locations sampled throughout the section, while retaining paired ATAC profiles at every location. `direction="R"` therefore creates a fragmented, spatially dispersed pattern of target-modality unregistration rather than a contiguous field-of-view loss. `missing='0'` denotes an RNA-unregistered location and `missing='1'` an observed ATAC-RNA pair.


In [ ]:
# Simulate global random missingness in target RNA
missing_indices = simulate_missing_sliding(adata_target, spatial_key="spatial", direction="R",
                                           missing_width=0.50, step_ratio=0.10, step_id=0,
                                           label_key="missing", lock_at_end=True, 
                                           point_size=4, plot=True, figsize=(4, 4))

print(f"RNA missing cells: {len(missing_indices)}/{adata_target.n_obs}")

### Preprocessing ATAC and RNA

ATAC and RNA are processed with their modality-specific workflows. ATAC is represented in latent semantic indexing (LSI) space to capture chromatin variation, whereas RNA remains the partially observed target. The randomly distributed `missing` labels are retained as an availability mask, ensuring that RNA-unregistered locations are not treated as observed molecular profiles during PRISM training.


In [ ]:
# Preprocess ATAC and RNA
adata_source, _ = preprocess_omics(adata_source, modality="ATAC", missing_key="missing", n_peak=10000,
                                   n_comps=50, data_role="source", use_lsi_as_X=True)

adata_target, _ = preprocess_omics(adata_target, modality="RNA", missing_key="missing", min_cells=10,
                                   hvgs=3000, data_role="target", compute_pca=False, save_log_layer=True,
                                   save_raw_eval=True)

print("ATAC shape after preprocessing:", adata_source.shape)
print("ATAC LSI shape:", adata_source.obsm["X_lsi"].shape)
print("RNA HVG shape after preprocessing:", adata_target.shape)

### Constructing the ATAC-LSI similarity prior

`compute_similarity_prior` derives an ATAC-LSI prior that supplies each RNA-unregistered location with chromatin-matched reference locations where RNA remains observed.

In [ ]:
distance_matrix, prior_metrics = compute_similarity_prior(adata_source, adata_target, PRIOR_PATH,
                                                          device=DEVICE, covet_k_spatial=6, covet_gene_num=None,
                                                          covet_use_layer=None, covet_use_obsm="X_lsi", 
                                                          spatial_key="spatial", missing_key="missing", 
                                                          store_neighbor_index=True)

In [ ]:
# Constructing spatial graphs
PRISM.Cal_Spatial_Net(adata_source, rad_cutoff=1.5)
PRISM.Stats_Spatial_Net(adata_source)
PRISM.Cal_Spatial_Net(adata_target, rad_cutoff=1.5)
PRISM.Stats_Spatial_Net(adata_target)

### Training PRISM

PRISM is trained on complete ATAC together with the randomly retained RNA profiles. It uses chromatin-matched context to impute RNA at dispersed unregistered locations while producing the joint embedding evaluated in the following tasks.

In [ ]:
# Train PRISM and add interaction principal components
adata_source_out, adata_target_out = PRISM.train_PRISM(adata_source, adata_target, distance_matrix,
                                                       k_top=5, n_epochs=1000, lr=1e-3,
                                                       output_dir=str(RESULTS_DIR), file_prefix=RUN_PREFIX, 
                                                       device=DEVICE, patience=20, min_epochs=50, 
                                                       center_drop_rate=0.1, noise=0.1,
                                                       load_model_path=False, interaction_pca=True)

### Task 1: Spatial-domain identification

In [ ]:
adata_clustered, domain_metrics = run_clustering_eval_plot(adata_source_out, emb_key="PRISM_emb", 
                                                           label_key="Combined_Clusters_annotation", 
                                                           cluster_key="PRISM_mclust", n_clusters=12, s=35,
                                                           use_pca=True, align_labels=True, 
                                                           aligned_key="PRISM_mclust_domain",
                                                           dataset_name="emb_mouse_e15.5")

### Task 2: RNA imputation

In [ ]:
# Evaluate RNA imputation on the top 800 HVGs
imputation_results = prism_eval_and_save(truth_adata=adata_target_out, adata=adata_target_out, 
                                         save_path=str(RESULTS_DIR), first_name=RUN_PREFIX, 
                                         missing_indices=missing_indices, topk_features=800,
                                         topk_rank_by="var_order", topk_only=True, save_topk_summary=True,
                                         save_files=False)

_ = plot_imputation_metric_boxplot(imputation_results, feature_names=adata_target.var_names,
                                   feature_label="RNA gene", output_suffix="RNA", plot_type="raincloud")

In [ ]:
# Visualize representative gene imputation
RNA_FEATURE_TO_PLOT = "ENSMUSG00000090386"
rna_spatial_plot = plot_prism_imputation_spatial(imputation_results=imputation_results, split1_indices=missing_indices,
                                                 feature=RNA_FEATURE_TO_PLOT, show_missing_only=False, 
                                                 highlight_missing=False, figsize=(8, 3))

feature_idx = adata_target.var_names.get_loc(RNA_FEATURE_TO_PLOT)
feature_metrics = {metric: round(float(imputation_results["raw"]["per_protein"][metric][feature_idx]), 4)
                   for metric in ("PCC", "SPCC", "MSE")}
print(f"Representative gene {RNA_FEATURE_TO_PLOT}: {feature_metrics}")